# Lab 6: Non-Linear Models and Feature Engineering

For this exercise, we will be using the titanic dataset. This dataset uses information about each passenger to predict if they survived or not.

In [ ]:
# import the libraries we need
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
import warnings
warnings.filterwarnings('ignore') #ignoring some deprication warnings

### Dataset

You can download the dataset from Kaggle ([link](https://www.kaggle.com/competitions/titanic/data?select=train.csv)) and use the train.csv or download the file from D2L.

In [ ]:
# Load the Titanic dataset into titanic_df
titanic_df = pd.read_csv('titanic.csv')
titanic_df

In [ ]:
# Get a summary of the dataset with .info
titanic_df.info()

In [ ]:
# Summary statistics for numerical columns
titanic_df.describe()

In [ ]:
# Count the number of missing values in each column
titanic_df.isnull().sum()

In [ ]:
# Visualize the distribution of passengers by class
class_colors = ['red', 'green', 'blue']

sns.countplot(x='Pclass', data=titanic_df, hue='Pclass')

plt.title('Passenger Class Distribution')
plt.show()

In [ ]:
# Visualize the distribution of passengers by gender
sns.countplot(x='Sex', data=titanic_df, hue='Sex')
plt.title('Passenger Gender Distribution')
plt.show()

In [ ]:
# Visualize the age distribution of passengers
sns.histplot(titanic_df['Age'].dropna(), kde=True, bins=20)
plt.title('Age Distribution')
plt.xlabel('Age')
plt.show()

In [ ]:
# Visualize the survival rate by class and gender
# Note: Survival -> 0 = No, 1 = Yes
sns.catplot(x='Pclass', hue='Sex', col='Survived', kind='count', data=titanic_df)
plt.suptitle('Survival by Class and Gender')
plt.show()

In [ ]:
# Remove any uniformative features and save the new dataset into titanic_df
titanic_df = titanic_df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'])

# drop all nulls
titanic_df.dropna(inplace=True)

In [ ]:
# check null values and data types
titanic_df.info()

In [ ]:
# Convert categorical variables to numerical using one-hot encoding
titanic_df = pd.get_dummies(titanic_df, columns=['Sex'], drop_first=True)
titanic_df = pd.get_dummies(titanic_df, columns=['Embarked'])
titanic_df

In [ ]:
# Visualize the correlation between numerical variables
sns.heatmap(titanic_df.corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Heatmap')
plt.show()

In [ ]:
# Split data into features (X) and target (y)
X = titanic_df.drop('Survived', axis=1)
y = titanic_df['Survived']

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Split the data into training and validation sets. Use 20% for validation and random_state=0
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=0)

# Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)

In [ ]:
from sklearn.svm import SVC
# Create an SVM classifier
svm = SVC()

# Fit the SVM model to the training data
svm.fit(X_train, y_train)


In [ ]:
# build a function that prints accuracy score, classification report,
# and the confusion matrix
def evaluate_model(y_val, y_preds):
    """
    create metrics for evaluating each model
    """

    # calculate accuracy score
    acc = accuracy_score(y_val, y_preds)

    # create classification report
    classification_repo = classification_report(y_val, y_preds)

    # confusion matrix
    cm = confusion_matrix(y_val, y_preds)

    # Evaluate the model
    print("Accuracy:", acc)
    print("Classification Report:\n", classification_repo)
    print("Confusion Matrix:\n", cm)


In [ ]:
# Predict the target for the validation data - store in y_pred_svm
y_pred_svm = svm.predict(X_val)

# evaluate the model
evaluate_model(y_val, y_pred_svm)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
# create a decision tree
decision_tree = DecisionTreeClassifier(random_state=0)

# fit the tree
decision_tree.fit(X_train, y_train)

# Predict the target for the validation data - store in y_pred_dt
y_pred_dt = decision_tree.predict(X_val)

In [ ]:
# evaluate the model
evaluate_model(y_val, y_pred_dt)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# build a random forest model
model = RandomForestClassifier(n_estimators=100, random_state=0)

# fit the model
model.fit(X_train, y_train)

# Predict the target for the validation data - store in y_pred_rf
y_pred_rf = model.predict(X_val)

In [ ]:
# evaluate the model
evaluate_model(y_val, y_pred_dt)

In [ ]:
# Create confusion matrices for each model
cm_dt = confusion_matrix(y_val, y_pred_dt)
cm_svm = confusion_matrix(y_val, y_pred_svm)
cm_rf = confusion_matrix(y_val, y_pred_rf)

# Set up subplots for each confusion matrix
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Create a heatmap for Decision Trees
sns.heatmap(cm_dt, annot=True, fmt='d', cmap='Blues', ax=axes[0])
axes[0].set_title('Decision Trees')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Create a heatmap for SVM
sns.heatmap(cm_svm, annot=True, fmt='d', cmap='Oranges', ax=axes[1])
axes[1].set_title('Support Vector Machine')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

# Create a heatmap for Random Forest
sns.heatmap(cm_rf, annot=True, fmt='d', cmap='Greens', ax=axes[2])
axes[2].set_title('Random Forest')
axes[2].set_xlabel('Predicted')
axes[2].set_ylabel('Actual')

plt.tight_layout()
plt.show()


In [ ]:
# Try to change the number of trees in random forest and see what happens

# Define a range of values for the number of trees (n_estimators)
n_estimators_values = [10, 50, 100]

# Lists to store confusion matrices for different n_estimators values
confusion_matrices = []

# Loop through different n_estimators values
for n_estimators in n_estimators_values:
    # Create and train the Random Forest model
    rf = RandomForestClassifier(n_estimators=n_estimators, random_state=0)
    rf.fit(X_train, y_train)

    # Make predictions on the test data
    y_pred_rf = rf.predict(X_val)

    # Calculate the confusion matrix and store it in the list
    cm = confusion_matrix(y_val, y_pred_rf)
    confusion_matrices.append(cm)

# Plot the confusion matrices using heatmaps
plt.figure(figsize=(15, 10))
for i, n_estimators in enumerate(n_estimators_values):
    plt.subplot(2, 3, i + 1)
    sns.heatmap(confusion_matrices[i], annot=True, fmt='d', cmap='Blues')
    plt.title(f'Confusion Matrix (n_estimators={n_estimators})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

plt.tight_layout()
plt.show()
